# Experiment 3 - Median Filter: Kernel Size and Stride

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from numpy.lib.stride_tricks import sliding_window_view
from skimage import data

In [ ]:
img = cv2.cvtColor(data.chelsea(), cv2.COLOR_RGB2GRAY)
rng = np.random.default_rng(42)

def salt_pepper(src, amount):
    out = src.copy()
    r = rng.random(src.shape)
    out[r < amount / 2] = 0
    out[r > 1 - amount / 2] = 255
    return out

noisy = salt_pepper(img, 0.3)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, im, t in zip(axes, [img, noisy], ["original", "noise 30%"]):
    ax.imshow(im, cmap="gray")
    ax.set_title(t)
    ax.axis("off")
plt.show()

In [ ]:
def windows(src, k, stride):
    p = k // 2
    padded = np.pad(src, p, mode="edge")
    return sliding_window_view(padded, (k, k))[::stride, ::stride]

def median_filter(src, k=3, stride=1):
    return np.median(windows(src, k, stride), axis=(-2, -1)).astype(np.uint8)

In [ ]:
def mean_filter(src, k=3, stride=1):
    return windows(src, k, stride).mean(axis=(-2, -1)).astype(np.uint8)

med = median_filter(noisy, 3)
avg = mean_filter(noisy, 3)

def psnr(a, b):
    mse = np.mean((a.astype(np.float64) - b.astype(np.float64)) ** 2)
    return 10 * np.log10(255 ** 2 / mse)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, im, t in zip(axes, [noisy, avg, med], ["noisy", "mean 3x3", "median 3x3"]):
    ax.imshow(im, cmap="gray")
    ax.set_title(f"{t}  psnr={psnr(img, im):.2f}")
    ax.axis("off")
plt.show()

In [ ]:
ks = [3, 5, 9]
fig, axes = plt.subplots(1, len(ks), figsize=(16, 4))
for ax, k in zip(axes, ks):
    out = median_filter(noisy, k)
    ax.imshow(out, cmap="gray")
    ax.set_title(f"k={k}x{k}  psnr={psnr(img, out):.2f}")
    ax.axis("off")
plt.show()

In [ ]:
strides = [1, 2, 4]
fig, axes = plt.subplots(1, len(strides), figsize=(16, 4))
for ax, s in zip(axes, strides):
    out = median_filter(noisy, 3, s)
    print(f"stride {s}: {out.shape}")
    ax.imshow(out, cmap="gray")
    ax.set_title(f"stride={s} {out.shape}")
    ax.axis("off")
plt.show()

In [ ]:
ref = cv2.medianBlur(noisy, 5)
mine = median_filter(noisy, 5)
print("matches cv2.medianBlur:", np.array_equal(ref, mine))
print("max abs diff:", int(np.abs(ref.astype(int) - mine.astype(int)).max()))